# CatTSunami Tutorial

In [ ]:
from ase.io import read
from ase.optimize import BFGS
import matplotlib.pyplot as plt

from fairchem.core.common.relaxation.ase_utils import OCPCalculator
from fairchem.core.models.model_registry import model_name_to_local_file
from fairchem.data.oc.core import Slab, Adsorbate, Bulk, AdsorbateSlabConfig

from ts_calc import Reaction
from ts_calc import OCPNEB
from ts_calc.autoframe import AutoFrameCoulping
from data import ADSORBATE_PKL_PATH, BULK_PKL_PATH, COUPLING_REACTION_DB_PATH

#Optional
from x3dase.x3d import X3D
from x3dase.visualize import view_x3d_n

from ase.mep import NEB

from get_activation_energy_from_path import get_activation_energy_from_path

#Set random seed
import numpy as np
np.random.seed(22)

In [ ]:
checkpoint_path = model_name_to_local_file('EquiformerV2-31M-S2EF-OC20-All+MD', local_cache='/data/ocp_checkpoints/')
cpu = False
calc = OCPCalculator(checkpoint_path = checkpoint_path, cpu = cpu, seed = 42)

In [ ]:
activation_energy = get_activation_energy_from_path("results/C4H4/bulk_Cu2Ta2/slab_003_100_bottom/4.traj", calc, plot=True)
print(f"Activation energy: {activation_energy:.3f} eV")

## Do enumerations in an AdsorbML style

In [ ]:
import pickle
with open(COUPLING_REACTION_DB_PATH, 'rb') as f:
    data = pickle.load(f)

# Adsorbate list
https://dl.fbaipublicfiles.com/opencatalystproject/data/input_generation/mapping_adsorbates_2020may12.txt

# Dimerization of acetylene

### Create product molecule

In [ ]:
a = np.array([12.6481, 0.0000, 0.0000])
b = np.array([-6.32405, 10.9535759, 0.0000])
c = np.array([0.0000, 0.0000, 25.0000])

def frac_to_cart(f1, f2, f3):
    return f1*a + f2*b + f3*c


coords = [
    (0.3651641561, 0.6358304085, 0.3979342337),
    (0.4087510231, 0.6002458535, 0.4404036923),
    (0.4769432915, 0.5373353659, 0.4411935485),
    (0.5191467173, 0.4950103886, 0.4001988170),
    (0.3148772704, 0.6813036621, 0.4109927235),
    (0.3894930450, 0.6225921391, 0.4802542760),
    (0.4976517302, 0.5200731886, 0.4817838547),
    (0.5690682096, 0.4515706363, 0.4160437167)
]

# Compute fractional centroid
centroid = np.mean(coords, axis=0)

# Center around (0,0,0)
centered_coords = coords - centroid

with open("data/*CHCHCH*CH.xyz", "w") as f:
    f.write(f"{len(coords)}\n")
    f.write("Converted from VASP Direct coords of *C4H5+2*H in DFT calculation model of reaction pathway model of 100%H of in SI of 'Selective electrosynthesis of 1,3-butadiene by tailoring the coverage of acetylene and water' ~Cheng.\n")
    for i, (f1, f2, f3) in enumerate(centered_coords):
        x, y, z = frac_to_cart(f1, f2, f3)
        element = "C" if i < 4 else "H"
        f.write(f"{element} {x:.6f} {y:.6f} {z:.6f}\n")

### Load product molecule and add it to database of adsorbants

In [ ]:
import numpy as np
import pickle

with open(ADSORBATE_PKL_PATH, 'rb') as f:
    ads_db = pickle.load(f)

new_ads = read('data/*CHCHCH*CH.xyz')
ads_db[87] = (new_ads, '*CHCHCH*CH', np.array([1, 2], dtype=int), '')

with open(ADSORBATE_PKL_PATH, "wb") as f:
    pickle.dump(ads_db, f)

### Create reaction object for this reaction using the new adsorbate created above

In [ ]:
new_react_C_H = {
    'reaction': "*CH*CH + *CH*CH -> *CHCHCH*CH",
    'reaction_type': 'coupling',
    'reactant1': 25,
    'reactant2': 25,
    'product': 87,
    'idx_mapping':[
        {0:1, 1:0, 2:5, 3:4, 4:2, 5:3, 6:6, 7:7},
        {0:1, 1:0, 2:5, 3:4, 4:3, 5:2, 6:7, 7:6},
        {0:0, 1:1, 2:4, 3:5, 4:2, 5:3, 6:6, 7:7},
        {0:0, 1:1, 2:4, 3:5, 4:3, 5:2, 6:7, 7:6},
        {0:2, 1:3, 2:6, 3:7, 4:0, 5:1, 6:4, 7:5},
        {0:2, 1:3, 2:6, 3:7, 4:1, 5:0, 6:5, 7:4},
        {0:3, 1:2, 2:7, 3:6, 4:0, 5:1, 6:4, 7:5},
        {0:3, 1:2, 2:7, 3:6, 4:1, 5:0, 6:5, 7:4},
    ],
    'edge_indices_initial': [
        (0, 1), (0, 2), (1, 3), (4,5), (4,6), (5, 7),
    ],
    'edge_indices_final': [
        (0, 1), (0, 4), (1, 2), (1,5), (2, 3), (2, 6), (3, 7),
    ],
}

# Reaction of acetylene with C2H3

### Create product molecule

In [ ]:
coords = [
    (0.4007064063872966, 0.6601531997019250, 0.3999214640045461),
    (0.4697168684003603, 0.6392601851885062, 0.4349023187746488),
    (0.5274233693040795, 0.5654873563746250, 0.4231969315712561),
    (0.6495727628367115, 0.6064693436735629, 0.4249172734998876),
    (0.3590133088606262, 0.7110710795558315, 0.4163513554585523),
    (0.4830429597335848, 0.6748191890805245, 0.4759013237037902),
    (0.4644515083142344, 0.4668336189121568, 0.4185184438249951),
    (0.6838080944179603, 0.5427646552261383, 0.4233549329736155),
    (0.7149600135733853, 0.7014591338416665, 0.4342037582929171)
]

# Compute fractional centroid
centroid = np.mean(coords, axis=0)

# Center around (0,0,0)
centered_coords = coords - centroid

with open("data/*CHCHCHCH2.xyz", "w") as f:
    f.write(f"{len(coords)}\n")
    f.write("Converted from VASP Direct coords of *C2H5+*H in DFT calculation model of reaction pathway model of 100%H of in SI of 'Selective electrosynthesis of 1,3-butadiene by tailoring the coverage of acetylene and water' ~Cheng.\n")
    for i, (f1, f2, f3) in enumerate(centered_coords):
        x, y, z = frac_to_cart(f1, f2, f3)
        element = "C" if i < 4 else "H"
        f.write(f"{element} {x:.6f} {y:.6f} {z:.6f}\n")

### Load product molecule and add it to database of adsorbants

In [ ]:
with open(ADSORBATE_PKL_PATH, 'rb') as f:
    ads_db = pickle.load(f)

new_ads = read('data/*CHCHCHCH2.xyz')
ads_db[88] = (new_ads, '*CHCHCHCH2', np.array([1, 2], dtype=int), '')

with open(ADSORBATE_PKL_PATH, "wb") as f:
    pickle.dump(ads_db, f)

### Create reaction object for this reaction using the new adsorbate created above

In [ ]:
new_react_C_H_1 = {
    'reaction': "*CH*CH + *CHCH2 -> *CHCHCHCH2",
    'reaction_type': 'coupling',
    'reactant1': 25,
    'reactant2': 34,
    'product': 88,
    'idx_mapping':[
        {0:0, 1:1, 2:4, 3:5, 4:2, 5:3, 6:6, 7:7, 8:8},
        {0:0, 1:1, 2:4, 3:5, 4:2, 5:3, 6:6, 7:8, 8:7},
        {0:1, 1:0, 2:5, 3:4, 4:2, 5:3, 6:6, 7:7, 8:8},
        {0:1, 1:0, 2:5, 3:4, 4:2, 5:3, 6:6, 7:8, 8:7},
    ],
    'edge_indices_initial': [
        (0, 1), (0, 2), (1, 3), (4,5), (4,6), (5, 7), (5, 8),
    ],
    'edge_indices_final': [
        (0, 1), (0, 4), (1, 2), (1,5), (2, 3), (2, 6), (3, 7), (3, 8),
    ],
}

# Dimerization of C2H3

### Create product molecule

In [ ]:
coords = [
    (0.3399866225772834, 0.6104782859881864, 0.4195626236909236),
    (0.4635744476286159, 0.6523548685866516, 0.4188823039400263),
    (0.5163182181855774, 0.5736257234435779, 0.4209871174337280),
    (0.6396739998111468, 0.6169494380997943, 0.4237754671983569),
    (0.3069164778359280, 0.6746946449974676, 0.4241601304042401),
    (0.2739001101582645, 0.5139211489302288, 0.4250053964590864),
    (0.5269636220503601, 0.7509028104879019, 0.4205296469005981),
    (0.4528853776570580, 0.4754828850814415, 0.4238931979776803),
    (0.7036790546358692, 0.7145181549267293, 0.4277811094902677),
    (0.6748021963244285, 0.5551814719033809, 0.4305717992581775)
]

# Compute fractional centroid
centroid = np.mean(coords, axis=0)

# Center around (0,0,0)
centered_coords = coords - centroid

with open("data/*CH2*CH*CH*CH2.xyz", "w") as f:
    f.write(f"{len(coords)}\n")
    f.write("Converted from VASP Direct coords of *C4H6 in DFT calculation model of reaction pathway model of 100%H of in SI of 'Selective electrosynthesis of 1,3-butadiene by tailoring the coverage of acetylene and water' ~Cheng.\n")
    for i, (f1, f2, f3) in enumerate(centered_coords):
        x, y, z = frac_to_cart(f1, f2, f3)
        element = "C" if i < 4 else "H"
        f.write(f"{element} {x:.6f} {y:.6f} {z:.6f}\n")

### Load product molecule and add it to database of adsorbants

In [ ]:
with open(ADSORBATE_PKL_PATH, 'rb') as f:
    ads_db = pickle.load(f)

new_ads = read('data/*CH2*CH*CH*CH2.xyz')
ads_db[89] = (new_ads, '*CH2*CH*CH*CH2', np.array([1, 2], dtype=int), '')

with open(ADSORBATE_PKL_PATH, "wb") as f:
    pickle.dump(ads_db, f)

### Create reaction object for this reaction using the new adsorbate created above

In [ ]:
new_react_C_H_2 = {
    'reaction': "*CHCH2 + *CHCH2 -> *CH2*CH*CH*CH2",
    'reaction_type': 'coupling',
    'reactant1': 34,
    'reactant2': 34,
    'product': 89,
    'idx_mapping':[
        {0:1, 1:0, 2:6, 3:4, 4:5, 5:2, 6:3, 7:7, 8:9, 9:8},
        {0:1, 1:0, 2:6, 3:5, 4:4, 5:2, 6:3, 7:7, 8:9, 9:8},
        {0:1, 1:0, 2:6, 3:4, 4:5, 5:2, 6:3, 7:7, 8:8, 9:9},
        {0:1, 1:0, 2:6, 3:5, 4:4, 5:2, 6:3, 7:7, 8:8, 9:9},
        {0:2, 1:3, 2:7, 3:9, 4:8, 5:1, 6:0, 7:6, 8:4, 9:5},
        {0:2, 1:3, 2:7, 3:8, 4:9, 5:1, 6:0, 7:6, 8:4, 9:5},
        {0:2, 1:3, 2:7, 3:9, 4:8, 5:1, 6:0, 7:6, 8:5, 9:4},
        {0:2, 1:3, 2:7, 3:8, 4:9, 5:1, 6:0, 7:6, 8:5, 9:4},
    ],
    'edge_indices_initial': [
        (0, 1), (0, 2), (1, 3), (1, 4), (5, 6), (5, 7), (6, 8), (6, 9),
    ],
    'edge_indices_final': [
        (0, 1), (0, 4), (0, 5), (1, 6), (1, 2), (2, 3), (2, 7), (3, 8), (3, 9), 
    ],
}

In [ ]:
import pickle
with open(COUPLING_REACTION_DB_PATH, 'rb') as f:
    data = pickle.load(f)

data.append(new_react_C_H)
data.append(new_react_C_H_1)
data.append(new_react_C_H_2)

with open(COUPLING_REACTION_DB_PATH, 'wb') as f:
    pickle.dump(data, f)


### exp from here (Past)

In [ ]:
from ase.io import read
from ase.optimize import BFGS, MDMin
import matplotlib.pyplot as plt

from fairchem.core.common.relaxation.ase_utils import OCPCalculator
from fairchem.core.models.model_registry import model_name_to_local_file
from fairchem.data.oc.core import Slab, Adsorbate, Bulk, AdsorbateSlabConfig

from ts_calc import Reaction
from ts_calc import OCPNEB
from ts_calc.autoframe import AutoFrameCoulping
from data import ADSORBATE_PKL_PATH, BULK_PKL_PATH, COUPLING_REACTION_DB_PATH

#Optional
from x3dase.x3d import X3D
from x3dase.visualize import view_x3d_n

#Set random seed
import numpy as np
np.random.seed(22)

In [ ]:
# Instantiate the reaction class for the reaction of interest
reaction = Reaction(reaction_str_from_db="*CH*CH + *CH*CH -> *CHCHCH*CH",
                    reaction_db_path=COUPLING_REACTION_DB_PATH,
                    adsorbate_db_path=ADSORBATE_PKL_PATH)
reaction.reaction_str_from_db

In [ ]:
# Instantiate our adsorbate class for the reactant and product
reactant1 = Adsorbate(adsorbate_id_from_db=reaction.reactant1_idx, adsorbate_db_path=ADSORBATE_PKL_PATH)
reactant2 = Adsorbate(adsorbate_id_from_db=reaction.reactant2_idx, adsorbate_db_path=ADSORBATE_PKL_PATH)
product1 = Adsorbate(adsorbate_id_from_db=reaction.product1_idx, adsorbate_db_path=ADSORBATE_PKL_PATH)

In [ ]:
# Grab the bulk and cut the slab we are interested in
bulk = Bulk(bulk_src_id_from_db="mp-30", bulk_db_path=BULK_PKL_PATH)
# slabs = bulk.get_slabs(max_miller=1)
slabs = Slab.from_bulk_get_specific_millers(bulk = bulk, specific_millers=(1,1,1))
all_slabs = bulk.get_slabs(max_miller=1)
all_slabs[0].millers

In [ ]:
# Perform site enumeration
# For AdsorbML num_sites = 100, but we use 5 here for brevity. This should be increased for practical use.
reactant1_configs = AdsorbateSlabConfig(slab = slabs[0], adsorbate = reactant1,
                                       mode="random_site_heuristic_placement",
                                       num_sites = 50).atoms_list
reactant2_configs = AdsorbateSlabConfig(slab = slabs[0], adsorbate = reactant2,
                                      mode="random_site_heuristic_placement",
                                      num_sites = 50).atoms_list
product1_configs = AdsorbateSlabConfig(slab = slabs[0], adsorbate = product1,
                                      mode="random_site_heuristic_placement",
                                      num_sites = 50).atoms_list

In [ ]:
from ase.visualize import view
i=2

view(reactant2_configs[i], viewer="x3d")


In [ ]:
# Instantiate the calculator
# NOTE: If you have a GPU, use cpu = False
# NOTE: Change the checkpoint path to locally downloaded files as needed
checkpoint_path = model_name_to_local_file('EquiformerV2-31M-S2EF-OC20-All+MD', local_cache='/data/ocp_checkpoints/')
cpu = False
calc = OCPCalculator(checkpoint_path = checkpoint_path, cpu = cpu, seed = 42)

In [ ]:
checkpoint_path

In [ ]:
# Relax the reactant1 systems
reactant1_energies = []
for config in reactant1_configs:
    config.calc = calc
    opt = BFGS(config, trajectory=None, logfile=None)
    opt.run(fmax = 0.05, steps=200)
    reactant1_energies.append(config.get_potential_energy())

In [ ]:
# Relax the reactant2 systems
reactant2_energies = []
for config in reactant2_configs:
    config.calc = calc
    opt = BFGS(config, logfile=None, trajectory=None)
    opt.run(fmax = 0.05, steps=200)
    reactant2_energies.append(config.get_potential_energy())

In [ ]:
# Relax the product1 systems
product1_energies = []
for config in product1_configs:
    config.calc = calc
    opt = BFGS(config, logfile=None, trajectory=None)
    opt.run(fmax = 0.05, steps=200)
    product1_energies.append(config.get_potential_energy())

In [ ]:
from ase.visualize import view
view(product1_configs[10], viewer="x3d")

## Dump the relaxed objects in a folder

In [ ]:
import pickle
product = "C4H4"
placement = "random_heuristic_50_50_50"
with open(f'data/debugging_NEB/{product}/{placement}/reactant1_configs.pkl', 'wb') as f:
    pickle.dump(reactant1_configs, f)

with open(f'data/debugging_NEB/{product}/{placement}/reactant1_energies.pkl', 'wb') as f:
    pickle.dump(reactant1_energies, f)

with open(f'data/debugging_NEB/{product}/{placement}/reactant2_configs.pkl', 'wb') as f:
    pickle.dump(reactant2_configs, f)

with open(f'data/debugging_NEB/{product}/{placement}/reactant2_energies.pkl', 'wb') as f:
    pickle.dump(reactant2_energies, f)

with open(f'data/debugging_NEB/{product}/{placement}/product1_configs.pkl', 'wb') as f:
    pickle.dump(product1_configs, f)

with open(f'data/debugging_NEB/{product}/{placement}/product1_energies.pkl', 'wb') as f:
    pickle.dump(product1_energies, f)

# Exp from here

## Enumerate NEBs

### Load the relaxed structures

In [ ]:
from ase.io import read
from ase.optimize import BFGS
import matplotlib.pyplot as plt

from fairchem.core.common.relaxation.ase_utils import OCPCalculator
from fairchem.core.models.model_registry import model_name_to_local_file
from fairchem.data.oc.core import Slab, Adsorbate, Bulk, AdsorbateSlabConfig

from ts_calc import Reaction
from ts_calc import OCPNEB
from ts_calc.autoframe import AutoFrameCoulping
from data import ADSORBATE_PKL_PATH, BULK_PKL_PATH, COUPLING_REACTION_DB_PATH

#Optional
from x3dase.x3d import X3D
from x3dase.visualize import view_x3d_n

#Set random seed
import numpy as np
np.random.seed(22)

In [ ]:
# Instantiate the reaction class for the reaction of interest
reaction_str = "*CH*CH + *CH*CH -> *CHCHCH*CH"
reaction = Reaction(reaction_str_from_db=reaction_str,
                    reaction_db_path=COUPLING_REACTION_DB_PATH,
                    adsorbate_db_path=ADSORBATE_PKL_PATH)

In [ ]:
# Instantiate the calculator
# NOTE: If you have a GPU, use cpu = False
# NOTE: Change the checkpoint path to locally downloaded files as needed
checkpoint_path = model_name_to_local_file('EquiformerV2-31M-S2EF-OC20-All+MD', local_cache='/data/ocp_checkpoints/')
cpu = False
calc = OCPCalculator(checkpoint_path = checkpoint_path, cpu = cpu, seed = 42)

In [ ]:
import pickle

if reaction_str == "*CH*CH + *CH*CH -> *CHCHCH*CH":
    product = 'C4H4'
elif reaction_str == "*CH*CH + *CHCH2 -> *CHCHCHCH2":
    product = 'C4H5'
elif reaction_str == "*CHCH2 + *CHCH2 -> *CH2*CH*CH*CH2":
    product = 'C4H6'

with open(f'data/debugging_NEB/{product}/random_heuristic_50_50_50/reactant1_configs.pkl', 'rb') as f:
    reactant1_configs = pickle.load(f)

with open(f'data/debugging_NEB/{product}/random_heuristic_50_50_50/reactant1_energies.pkl', 'rb') as f:
    reactant1_energies = pickle.load(f)

with open(f'data/debugging_NEB/{product}/random_heuristic_50_50_50/reactant2_configs.pkl', 'rb') as f:
    reactant2_configs = pickle.load(f)

with open(f'data/debugging_NEB/{product}/random_heuristic_50_50_50/reactant2_energies.pkl', 'rb') as f:
    reactant2_energies = pickle.load(f)

with open(f'data/debugging_NEB/{product}/random_heuristic_50_50_50/product1_configs.pkl', 'rb') as f:
    product1_configs = pickle.load(f)

with open(f'data/debugging_NEB/{product}/random_heuristic_50_50_50/product1_energies.pkl', 'rb') as f:
    product1_energies = pickle.load(f)

if product == 'C4H4':
    dist_max, r_react_max = 5.0, 3.0


if product == 'C4H5':
    dist_max, r_react_max = 6.5, 3.0


if product == 'C4H6':
    dist_max, r_react_max = 7.0, 6.0

In [ ]:
af = AutoFrameCoulping(
            reaction = reaction,
            reactant1_systems = reactant1_configs,
            reactant1_energies = reactant1_energies,
            reactant2_systems = reactant2_configs,
            reactant2_energies = reactant2_energies,
            product_systems = product1_configs,
            product_energies = product1_energies,
            dist_max=dist_max,
            r_react_max=r_react_max,
            r_react_min=1.0,
)

In [ ]:
nframes = 10
frame_sets, mapping_idxs = af.get_neb_frames(
        calc, n_frames=nframes,
        n_initial_frames=5, n_final_frames_per_initial=1,
        idpp=True
)

In [ ]:
import pickle
with open('results_test/C4H4/bulk_Pt3Ir/slab_000_111/idpp/5.0_3.0/initial_NEBs.pkl', 'rb') as f:
    frame_sets = pickle.load(f)
len(frame_sets)

In [ ]:
len(frame_sets[0])

In [ ]:
from ase.io import write
write('GIFS_butadiene/init_neb_C4H5_50_placements_6_5dist_3react_3_idpp.gif', frame_sets[3], interval=400)

## Run NEBs

In [ ]:
# If you run the above cell -- dont run this one
from ase.optimize import LBFGS
from ts_calc import OCPNEB
from ase.optimize import FIRE
from main import SafeLBFGS


fmax = 0.05 # [eV / ang**2]
delta_fmax_climb = 0.4
neb = OCPNEB(
    frame_sets[0],
    checkpoint_path=checkpoint_path,
k=1,
batch_size=10,
cpu = False
)

optimizer = SafeLBFGS(
    neb,
    trajectory=f"test.traj",

)


conv = optimizer.run(fmax=fmax+delta_fmax_climb, steps=200)
if conv:
    print(f"Climbing the NEB with fmax = {fmax}")
    neb.climb = True
    conv = optimizer.run(fmax=fmax, steps=300)

if conv:
    print("The NEB has succesfully converged.")

In [ ]:
from time import time
import cProfile, pstats, io

pr = cProfile.Profile()
pr.enable()

t0 = time()
optimizer.step()   # single optimization step
print("Step wall time:", time()-t0)

pr.disable()
s = io.StringIO()
ps = pstats.Stats(pr, stream=s).sort_stats("cumtime")
ps.print_stats(20)   # top 20 functions by cumulative time
print(s.getvalue())


## Visualize the results

In [ ]:
from ase.io import read, write

neb_traj = read(f"trajectories/test_converged_C4H6_mp_30.traj", ":")
optimized_neb = neb_traj[-10:]
print(len(neb_traj))

In [ ]:
es  = []
for frame in optimized_neb:
    frame.set_calculator(calc)
    es.append(frame.get_potential_energy())

In [ ]:
from ase.visualize import view
print(es.index(max(es)))
view(optimized_neb[es.index(max(es))], viewer="x3d")

In [ ]:
from ase.io import read, write

neb_traj = read(f"results/C4H4/bulk_Ru2Sn2/slab_002_100/4.traj", ":")
optimized_neb = neb_traj[-10:]
print(len(neb_traj))

In [ ]:
from ase.io import write
write('GIFS_butadiene/C4H4/Ru2Sn2/100_top.gif', optimized_neb, interval=400)

In [ ]:
from ase.visualize import view
view(optimized_neb[-1], viewer="x3d")

In [ ]:
import pandas as pd
df = pd.read_csv('results_reduced/all_results.csv')
df.iloc[:30]

In [ ]:
best_bulk_rows = []
seen_bulks = set()

for _, row in df.iterrows():
    bulk = row['bulk']
    if bulk not in seen_bulks:
        best_bulk_rows.append(row)
        seen_bulks.add(bulk)
    if len(best_bulk_rows) == 30:
        break

# Convert back to DataFrame
best_bulk_df = pd.DataFrame(best_bulk_rows)

print(best_bulk_df)


In [ ]:
from fairchem.core.common.relaxation.ase_utils import OCPCalculator
from fairchem.core.models.model_registry import model_name_to_local_file

checkpoint_path = model_name_to_local_file('EquiformerV2-31M-S2EF-OC20-All+MD', local_cache='/data/ocp_checkpoints/')
cpu = False
calc = OCPCalculator(checkpoint_path = checkpoint_path, cpu = cpu, seed = 42)

In [ ]:
from ase.io import read, write
from get_activation_energy_from_path import get_activation_energy_from_path
import matplotlib.pyplot as plt
for idx, traj_path in enumerate(df['traj_path'][61:70], start=61):
    neb_traj = read(traj_path, ":")
    get_activation_energy_from_path(traj_path, calc, plot=True)
    plt.savefig(f'GIFS/butadiene/best_copper_alloys/{idx}.png')
    plt.close()
    write(f'GIFS/butadiene/best_copper_alloys/{idx}.gif', neb_traj, interval=400)
    plt.close()



In [ ]:
get_activation_energy_from_path('results_reduced/C4H6/bulk_Cu2Sn2/slab_003_100_bottom/1.traj', calc, plot=True)

In [ ]:
for idx, bulk in enumerate(df['bulk'][:61]):
    print(f"{idx}: {bulk}")

In [ ]:
list_of_reasonable_NEBs = [5,6,9,10,11,12,13,14,15,16,17,18,19,20,21,24,26,27,28,31,35,36,37,38,42,44,45,47,51,53,54,55,59,60,61,62,67,68]
set_good_copper_alloys = set()
for idx in list_of_reasonable_NEBs:
    bulk = df['bulk'][idx]
    set_good_copper_alloys.add(bulk)
print(set_good_copper_alloys)
print(len(set_good_copper_alloys))

In [ ]:
import pickle
with open("data/alloy/copper_alloy_atoms_10_jobs/bulks_screened_copper.pkl", 'rb') as f:
    copper_alloys = pickle.load(f)

copper_alloys_to_screen_for_C2H4 = []
for idx, bulk in enumerate(copper_alloys):
    if bulk['src_id'] in set_good_copper_alloys:
        copper_alloys_to_screen_for_C2H4.append(bulk)

with open("data/alloy/30_best_copper_alloy_atoms_10_jobs/bulks_screened_copper_C2H4.pkl", "wb") as f:
    pickle.dump(copper_alloys_to_screen_for_C2H4, f)

print(len(copper_alloys_to_screen_for_C2H4))
copper_alloys_to_screen_for_C2H4